# MDLM (Masked Diffusion Language Model) Training on Colab

GPU ランタイムを選択してから実行してください：  
**Runtime → Change runtime type → T4 GPU (or better)**

In [7]:
# 1. リポジトリをクローン（自分のリポジトリURLに変更してください）
!git clone https://github.com/tori3-po4/diffusion_language_model.git
%cd diffusion_language_model

fatal: destination path 'diffusion_language_model' already exists and is not an empty directory.
/content/diffusion_language_model/diffusion_language_model


In [8]:
# 2. uv のインストールと依存パッケージのインストール
!pip install uv
!uv pip install --system -r pyproject.toml

Using Python 3.12.12 environment at: /usr
Resolved 94 packages in 239ms                                        
Audited 94 packages in 1ms


In [9]:
# 3. GPU の確認
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
    print(f"BF16 support: {torch.cuda.is_bf16_supported()}")

CUDA available: True
Device: Tesla T4
Memory: 15.6 GB
BF16 support: True


In [10]:
# 4. T4 GPU は bf16 非対応なので、自動で fp16 にフォールバック
import yaml

with open("configs/colab.yaml") as f:
    config = yaml.safe_load(f)

if not torch.cuda.is_bf16_supported():
    print("BF16 not supported, switching to FP16")
    config["training"]["mixed_precision"] = "fp16"
    with open("configs/colab.yaml", "w") as f:
        yaml.dump(config, f, default_flow_style=False)
else:
    print("Using BF16 mixed precision")

Using BF16 mixed precision


In [11]:
# 5. 学習開始
!python src/train.py --config configs/colab.yaml

Model parameters: 35,851,346 (35.9M)
Training:   0% 0/200000 [00:00<?, ?it/s]
Resolving data files: 100% 2410/2410 [00:00<00:00, 44011.01it/s]
Training:   0% 0/200000 [00:07<?, ?it/s]
Traceback (most recent call last):
  File "/content/diffusion_language_model/diffusion_language_model/src/train.py", line 279, in <module>
    main()
  File "/content/diffusion_language_model/diffusion_language_model/src/train.py", line 220, in main
    loss = compute_loss(
           ^^^^^^^^^^^^^
  File "/content/diffusion_language_model/diffusion_language_model/src/train.py", line 73, in compute_loss
    logits = model(xt, sigma)
             ^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/torch/nn/modules/module.py", line 1776, in _wrapped_call_impl
    return self._call_impl(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/torch/nn/modules/module.py", line 1787, in _call_impl
    return forward_call(*args, **kwargs)
        

In [12]:
# 6. 学習曲線の表示
%matplotlib inline
import matplotlib
matplotlib.use("Agg")

from src.plot import load_metrics, smooth
import matplotlib.pyplot as plt

steps, losses, lrs = load_metrics("logs/metrics.csv")
if steps:
    smoothed = smooth(losses, 10)
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 6), sharex=True)
    ax1.plot(steps, losses, alpha=0.3, label="Raw")
    ax1.plot(steps, smoothed, linewidth=2, label="Smoothed")
    ax1.set_ylabel("Loss")
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    ax2.plot(steps, lrs, color="C1")
    ax2.set_ylabel("LR")
    ax2.set_xlabel("Step")
    ax2.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

ImportError: Cannot load backend 'TkAgg' which requires the 'tk' interactive framework, as 'headless' is currently running

In [ ]:
# 7. テキスト生成（チェックポイントのパスを指定）
!python src/sample.py --config configs/colab.yaml --checkpoint checkpoints/final --steps 256 --num_samples 4

In [ ]:
# 8. チェックポイントを Google Drive に保存（セッション切れ対策）
from google.colab import drive
drive.mount("/content/drive")
!cp -r checkpoints /content/drive/MyDrive/mdlm_checkpoints